In [1]:
import numpy as np
import pandas as pd
import os

DATA_DIR = "./data"  # wherever you saved them

In [2]:
df = pd.read_csv(f"{DATA_DIR}/userid_trackid_count.tsv", sep="\t",
                 names=["user_id", "track_id", "play_count"])

print(df.shape)           # (252M rows, 3 cols)
print(df.head())
print(df["play_count"].describe())

# Quick sparsity check
n_users = df["user_id"].nunique()
n_tracks = df["track_id"].nunique()
sparsity = 1 - len(df) / (n_users * n_tracks)
print(f"Users: {n_users}, Tracks: {n_tracks}, Sparsity: {sparsity:.4f}")

C:\Users\toshn\AppData\Local\Temp\ipykernel_7288\544487986.py:1: DtypeWarning: Columns (0: user_id, 1: play_count) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{DATA_DIR}/userid_trackid_count.tsv", sep="\t",


(50016043, 3)
   user_id          track_id play_count
0  user_id          track_id      count
1    92915  2fwmmHiKaZqDUlY0          1
2    92915  b4rHYoJ9wWDSZT3M          1
3    92915  LF0WX7weWu3LNbjC          2
4    92915  9AwpGifIAWQwGWZG          2
count     50016043
unique        2042
top              1
freq      22850088
Name: play_count, dtype: int64
Users: 119142, Tracks: 56513, Sparsity: 0.9926


In [9]:
def load_feature_tsv(path):
    df = pd.read_csv(path, sep="\t", header=0, index_col=0, low_memory=False)
    # Row 0 becomes the header, col 0 becomes the index (track_id)
    # Drop any lingering non-numeric rows (shouldn't be any after this)
    df = df.apply(pd.to_numeric, errors='coerce')
    
    track_ids = df.index.tolist()
    feature_matrix = df.values.astype(np.float32)
    return track_ids, feature_matrix

track_ids, mfcc = load_feature_tsv("data/id_mfcc_stats.tsv")
print(f"Tracks:  {len(track_ids)}")       # 109269
print(f"Shape:   {mfcc.shape}")           # (109269, 104)
print(f"Sample:  {mfcc[0][:5]}")
print(f"NaNs:    {np.isnan(mfcc).sum()}")

Tracks:  109269
Shape:   (109269, 104)
Sample:  [20.64096   -5.828413  -6.1408587  2.5266645 -1.4095522]
NaNs:    0


In [11]:
p1_files = {
    "mfcc_stats":   "id_mfcc_stats.tsv",
    "blf_spectral": "id_blf_spectral.tsv",
    "essentia":     "id_essentia.tsv",
    "ivec256":      "id_ivec256.tsv",
    "incp":         "id_incp.tsv",
    "genres_tfidf": "id_genres_tf-idf.tsv",
}

features = {}
for name, fname in p1_files.items():
    track_ids, mat = load_feature_tsv(f"data/{fname}")
    features[name] = {"track_ids": track_ids, "matrix": mat}
    print(f"{name:20s}  shape={mat.shape}  NaNs={np.isnan(mat).sum()}")

mfcc_stats            shape=(109269, 104)  NaNs=0
blf_spectral          shape=(109262, 980)  NaNs=0
essentia              shape=(109180, 1034)  NaNs=0
ivec256               shape=(109269, 100)  NaNs=0
incp                  shape=(98877, 4096)  NaNs=0
genres_tfidf          shape=(109269, 685)  NaNs=0


In [12]:
# Take mfcc_stats as the reference
ref_ids = features["mfcc_stats"]["track_ids"]

for name, data in features.items():
    match = data["track_ids"] == ref_ids
    print(f"{name:20s}  aligned={match}")

mfcc_stats            aligned=True
blf_spectral          aligned=False
essentia              aligned=False
ivec256               aligned=False
incp                  aligned=False
genres_tfidf          aligned=False


In [13]:
interactions = pd.read_csv("data/userid_trackid_count.tsv", sep="\t",
                            header=None, names=["user_id", "track_id", "play_count"])
print(interactions.shape)
print(interactions.head())
print(interactions["play_count"].describe())

# Cross-check: are all interacted track_ids present in feature files?
interacted_tracks = set(interactions["track_id"].unique())
feature_tracks    = set(ref_ids)
print(f"\nTracks in interactions not in features: {len(interacted_tracks - feature_tracks)}")
print(f"Tracks in features not in interactions: {len(feature_tracks - interacted_tracks)}")

C:\Users\toshn\AppData\Local\Temp\ipykernel_7288\2838057891.py:1: DtypeWarning: Columns (0: user_id, 1: play_count) have mixed types. Specify dtype option on import or set low_memory=False.
  interactions = pd.read_csv("data/userid_trackid_count.tsv", sep="\t",


(50016043, 3)
   user_id          track_id play_count
0  user_id          track_id      count
1    92915  2fwmmHiKaZqDUlY0          1
2    92915  b4rHYoJ9wWDSZT3M          1
3    92915  LF0WX7weWu3LNbjC          2
4    92915  9AwpGifIAWQwGWZG          2
count     50016043
unique        2042
top              1
freq      22850088
Name: play_count, dtype: int64

Tracks in interactions not in features: 1
Tracks in features not in interactions: 52757


In [15]:
DATA_DIR = "data"

# ── 1. Load interactions cleanly ──────────────────────────────────────────────
interactions = pd.read_csv(
    f"{DATA_DIR}/userid_trackid_count.tsv", sep="\t",
    header=None, names=["user_id", "track_id", "play_count"],
    dtype=str, low_memory=False
)

# Drop the leaked header row
interactions = interactions[interactions["user_id"] != "user_id"].copy()
interactions["play_count"] = interactions["play_count"].astype(int)
interactions["user_id"]    = interactions["user_id"].astype(int)

print(interactions.shape)           # should be (50016042, 3) — one less row
print(interactions.dtypes)
print(interactions["play_count"].describe())

# ── 2. Build the master track index ───────────────────────────────────────────
# Intersection of all feature files + tracks that appear in interactions
# Load just the index column of each file (fast, no feature data yet)

def get_track_ids(path):
    df = pd.read_csv(path, sep="\t", usecols=[0], header=0,
                     low_memory=False)
    return set(df.iloc[:, 0].tolist())

file_ids = {
    "mfcc_stats":   get_track_ids(f"{DATA_DIR}/id_mfcc_stats.tsv"),
    "blf_spectral": get_track_ids(f"{DATA_DIR}/id_blf_spectral.tsv"),
    "essentia":     get_track_ids(f"{DATA_DIR}/id_essentia.tsv"),
    "ivec256":      get_track_ids(f"{DATA_DIR}/id_ivec256.tsv"),
    "genres_tfidf": get_track_ids(f"{DATA_DIR}/id_genres_tf-idf.tsv"),
    # incp excluded from intersection — too many missing; handle separately
}

interacted_tracks = set(interactions["track_id"].unique())

# Core tracks: present in all feature files AND have at least one interaction
core_tracks = interacted_tracks.copy()
for name, ids in file_ids.items():
    before = len(core_tracks)
    core_tracks &= ids
    print(f"After intersecting {name:20s}: {len(core_tracks):>6} tracks (lost {before - len(core_tracks)})")

print(f"\nFinal core track count: {len(core_tracks)}")

# Build a stable sorted index: track_id → integer position
track_list  = sorted(core_tracks)           # deterministic order
track2idx   = {t: i for i, t in enumerate(track_list)}
print(f"track2idx built: {len(track2idx)} entries")

(50016042, 3)
user_id       int64
track_id        str
play_count    int64
dtype: object
count    5.001604e+07
mean     5.058065e+00
std      2.363959e+01
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      6.086300e+04
Name: play_count, dtype: float64
After intersecting mfcc_stats          :  56512 tracks (lost 0)
After intersecting blf_spectral        :  56508 tracks (lost 4)
After intersecting essentia            :  56460 tracks (lost 48)
After intersecting ivec256             :  56460 tracks (lost 0)
After intersecting genres_tfidf        :  56460 tracks (lost 0)

Final core track count: 56460
track2idx built: 56460 entries


In [17]:
def load_aligned(path, track2idx, chunksize=3000):
    """Load a feature TSV aligned to track2idx. Chunked to handle wide files."""
    
    # First pass: get column count
    header = pd.read_csv(path, sep="\t", nrows=0, header=0, low_memory=False)
    D = len(header.columns) - 1  # subtract track_id column
    N = len(track2idx)
    
    out = np.full((N, D), np.nan, dtype=np.float32)
    filled = 0

    for chunk in pd.read_csv(path, sep="\t", header=0, index_col=0,
                              chunksize=chunksize, low_memory=False):
        chunk = chunk.apply(pd.to_numeric, errors="coerce")
        for track_id, row in chunk.iterrows():
            tid = str(track_id)
            if tid in track2idx:
                out[track2idx[tid]] = row.values.astype(np.float32)
                filled += 1

    print(f"shape=({N}, {D})  filled={filled}/{N}  NaN rows={np.isnan(out).any(axis=1).sum()}", flush=True)
    return out

feature_arrays = {}
for name, fname in {
    "mfcc_stats":   "id_mfcc_stats.tsv",
    "blf_spectral": "id_blf_spectral.tsv",
    "essentia":     "id_essentia.tsv",
    "ivec256":      "id_ivec256.tsv",
    "genres_tfidf": "id_genres_tf-idf.tsv",
    "tags_tfidf":   "id_tags_tf-idf.tsv",
}.items():
    print(f"Loading {name}...", end=" ")
    arr = load_aligned(f"{DATA_DIR}/{fname}", track2idx)
    feature_arrays[name] = arr
    nan_rows = np.isnan(arr).any(axis=1).sum()
    print(f"shape={arr.shape}  NaN rows={nan_rows}")

Loading mfcc_stats... shape=(56460, 104)  filled=56460/56460  NaN rows=0
shape=(56460, 104)  NaN rows=0
Loading blf_spectral... shape=(56460, 980)  filled=56460/56460  NaN rows=0
shape=(56460, 980)  NaN rows=0
Loading essentia... shape=(56460, 1034)  filled=56460/56460  NaN rows=0
shape=(56460, 1034)  NaN rows=0
Loading ivec256... shape=(56460, 100)  filled=56460/56460  NaN rows=0
shape=(56460, 100)  NaN rows=0
Loading genres_tfidf... shape=(56460, 685)  filled=56460/56460  NaN rows=0
shape=(56460, 685)  NaN rows=0
Loading tags_tfidf... 

MemoryError: Unable to allocate 37.5 GiB for an array with shape (56460, 178108) and data type float32

In [18]:
import random

# ── Pick 5000 tracks common across all files (excluding tags_tfidf) ──────────
file_ids_no_tags = {
    "mfcc_stats":   get_track_ids(f"{DATA_DIR}/id_mfcc_stats.tsv"),
    "blf_spectral": get_track_ids(f"{DATA_DIR}/id_blf_spectral.tsv"),
    "essentia":     get_track_ids(f"{DATA_DIR}/id_essentia.tsv"),
    "ivec256":      get_track_ids(f"{DATA_DIR}/id_ivec256.tsv"),
    "genres_tfidf": get_track_ids(f"{DATA_DIR}/id_genres_tf-idf.tsv"),
    # tags_tfidf dropped — 178K cols, not usable as dense array
    # incp handled separately as optional modality
}

# Tracks present in all files AND in interactions
core_tracks = set(interactions["track_id"].unique())
for name, ids in file_ids_no_tags.items():
    core_tracks &= ids

# Sample 5000
random.seed(42)
sampled_tracks = random.sample(sorted(core_tracks), 5000)
track2idx      = {t: i for i, t in enumerate(sampled_tracks)}

print(f"Sampled {len(track2idx)} tracks from {len(core_tracks)} core tracks")

Sampled 5000 tracks from 56460 core tracks


In [19]:
def load_aligned_sample(path, track2idx, chunksize=3000):
    """Load only rows whose track_id is in track2idx."""
    header  = pd.read_csv(path, sep="\t", nrows=0, header=0, low_memory=False)
    D       = len(header.columns) - 1
    N       = len(track2idx)
    out     = np.zeros((N, D), dtype=np.float32)
    filled  = 0

    for chunk in pd.read_csv(path, sep="\t", header=0, index_col=0,
                              chunksize=chunksize, low_memory=False):
        chunk = chunk.apply(pd.to_numeric, errors="coerce")
        wanted = chunk.index.astype(str).isin(track2idx)
        if not wanted.any():
            continue
        for track_id, row in chunk[wanted].iterrows():
            out[track2idx[str(track_id)]] = row.values.astype(np.float32)
            filled += 1
        if filled == N:
            break   # got everything, stop reading

    print(f"shape={out.shape}  filled={filled}/{N}  NaN rows={np.isnan(out).any(axis=1).sum()}")
    return out

feature_arrays = {}
for name, fname in {
    "mfcc_stats":   "id_mfcc_stats.tsv",
    "blf_spectral": "id_blf_spectral.tsv",
    "essentia":     "id_essentia.tsv",
    "ivec256":      "id_ivec256.tsv",
    "genres_tfidf": "id_genres_tf-idf.tsv",
}.items():
    print(f"Loading {name}...", end=" ")
    feature_arrays[name] = load_aligned_sample(f"{DATA_DIR}/{fname}", track2idx)

Loading mfcc_stats... shape=(5000, 104)  filled=5000/5000  NaN rows=0
Loading blf_spectral... shape=(5000, 980)  filled=5000/5000  NaN rows=0
Loading essentia... shape=(5000, 1034)  filled=5000/5000  NaN rows=0
Loading ivec256... shape=(5000, 100)  filled=5000/5000  NaN rows=0
Loading genres_tfidf... shape=(5000, 685)  filled=5000/5000  NaN rows=0


In [20]:
print("Loading incp...", end=" ")
df_incp    = pd.read_csv(f"{DATA_DIR}/id_incp.tsv", sep="\t", header=0,
                          index_col=0, low_memory=False)
df_incp    = df_incp.apply(pd.to_numeric, errors="coerce")

N          = len(track2idx)
incp_array = np.zeros((N, 4096), dtype=np.float32)
incp_mask  = np.zeros(N, dtype=bool)

for track_id, idx in track2idx.items():
    if track_id in df_incp.index:
        incp_array[idx] = df_incp.loc[track_id].values.astype(np.float32)
        incp_mask[idx]  = True

feature_arrays["incp"]      = incp_array
feature_arrays["incp_mask"] = incp_mask
print(f"shape={incp_array.shape}  coverage={incp_mask.sum()}/{N} ({incp_mask.mean()*100:.1f}%)")

Loading incp... shape=(5000, 4096)  coverage=4565/5000 (91.3%)


In [24]:
interactions_sample = interactions[interactions["track_id"].isin(track2idx)].copy()
interactions_sample["track_idx"] = interactions_sample["track_id"].map(track2idx)

user_list  = sorted(interactions_sample["user_id"].unique())
user2idx   = {u: i for i, u in enumerate(user_list)}
interactions_sample["user_idx"] = interactions_sample["user_id"].map(user2idx)

print(f"Interactions: {len(interactions_sample):,}")
print(f"Users: {len(user2idx):,}   Tracks: {len(track2idx):,}")

# Save
import os, json
os.makedirs("data/processed", exist_ok=True)

for name, arr in feature_arrays.items():
    np.save(f"data/processed/{name}.npy", arr)

with open("data/processed/track2idx.json", "w") as f:
    json.dump(track2idx, f)
with open("data/processed/user2idx.json", "w") as f:
    json.dump({str(k): v for k, v in user2idx.items()}, f)

interactions_sample[["user_idx", "track_idx", "play_count"]].to_csv(
    "data/processed/interactions.csv", index=False
)
print("All saved to data/processed/")

Interactions: 4,461,696
Users: 111,723   Tracks: 5,000
All saved to data/processed/
